# Building Models in PyTorch

This notebook introduces building neural networks in PyTorch. 
We start with simple linear models on dummy data, move to image inputs, 
introduce convolutional layers, and briefly explore example of more advanced architecture (autoencoders).

The goal is to **understand model construction, forward pass, tensor shapes, and feature maps**.

## What are Features?

- In machine learning, a **feature** is an individual measurable property of your data.
- For tabular data, features are usually columns (e.g., height, weight, age).
- For images, features can be:
  - Raw pixel values
  - Higher-level representations extracted by **convolutional layers** (feature maps)
- Neural networks automatically learn **useful features** for the task during training.
- Each layer in a network transforms its input into a new set of features.

> When inspecting tensors after each layer, think of the channel dimension as "different features" extracted by the network.

## Section 1 – Simple Linear Model

We define a linear model manually using PyTorch tensors. This demonstrates 
forward computation and parameter shapes.

In [1]:
import torch

# Example tabular dataset: apartment features
# features: [size_m2, rooms, distance_to_center_km, floor, building_age]

X = torch.tensor([
    [50, 2, 3.5, 4, 10],   # apartment 1
    [70, 3, 5.0, 2, 20],   # apartment 2
    [35, 1, 1.2, 5, 5],    # apartment 3
    [90, 4, 8.0, 1, 30]    # apartment 4
]).float()

# Dummy target: apartment price (in thousands)
y = torch.tensor([
    [410],
    [680],
    [213],
    [660],
]).float()

batch_size, features = X.shape
output_dim = 1

print("Input features:")
print(X)

print("\nTarget prices:")
print(y)

# Initialize weights manually
W = torch.randn(features, output_dim, requires_grad=True)
b = torch.zeros(output_dim, requires_grad=True)

# Forward pass (linear model)
y_pred = X @ W + b

print("\nModel prediction:")
print(y_pred)

print("\nPrediction shape:", y_pred.shape)

print("\nWeights:", W)

Input features:
tensor([[50.0000,  2.0000,  3.5000,  4.0000, 10.0000],
        [70.0000,  3.0000,  5.0000,  2.0000, 20.0000],
        [35.0000,  1.0000,  1.2000,  5.0000,  5.0000],
        [90.0000,  4.0000,  8.0000,  1.0000, 30.0000]])

Target prices:
tensor([[410.],
        [680.],
        [213.],
        [660.]])

Model prediction:
tensor([[21.1575],
        [37.3986],
        [12.6934],
        [53.4382]], grad_fn=<AddBackward0>)

Prediction shape: torch.Size([4, 1])

Weights: tensor([[ 0.1204],
        [ 0.4132],
        [-0.3000],
        [ 0.2486],
        [ 1.4367]], requires_grad=True)


> A linear layer is simply a weighted sum of input features plus a bias.

```
price =
w1 * size +
w2 * rooms +
w3 * distance +
w4 * floor +
w5 * age +
bias
```

## Section 3 – Linear Model with `nn.Module`

Instead of manual weights, we use `nn.Linear`. This is the standard PyTorch way.

In [ ]:
import torch.nn as nn

linear_model = nn.Linear(features, output_dim)
print(linear_model)

# Forward pass on dummy data
y_pred2 = linear_model(X)
print("Output shape:", y_pred2.shape)

print(linear_model.state_dict())

Linear(in_features=5, out_features=1, bias=True)
Output shape: torch.Size([4, 1])
OrderedDict([('weight', tensor([[-0.0634, -0.0779,  0.4355, -0.3311,  0.1745]])), ('bias', tensor([-0.0168]))])


: 

### From Single-Layer to Deep Neural Networks

Single-layer models are simple:

```python
linear_model = nn.Linear(features, output_dim)  # one layer
```

For more complex tasks, we use **Deep Neural Networks** – multiple layers stacked together.

In PyTorch, the standard way is to define a class inheriting from ```nn.Module```:

```python
import torch.nn as nn
import torch.nn.functional as F

class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Define layers
        self.fc1 = nn.Linear(10, 32)  # first hidden layer
        self.fc2 = nn.Linear(32, 16)  # second hidden layer
        self.fc3 = nn.Linear(16, 2)   # output layer

    def forward(self, x):
        # Define how data flows
        x = F.relu(self.fc1(x))       # activation after first layer
        x = F.relu(self.fc2(x))       # activation after second layer
        x = self.fc3(x)               # raw output (logits)
        return x

```



- Each layer learns a different feature representation of the input.
- forward defines the computation graph.
- Using classes allows flexible architectures with many layers.

## Section 4 – Linear Model on Image Data

Images have shape `[batch_size, channels, height, width]`. 
We flatten them to feed into a linear layer.

In [ ]:
# Dummy images: batch_size=2, grayscale 32x32
dummy_images = torch.randn(2, 1, 32, 32)

# Flatten for linear layer
dummy_images_flat = dummy_images.view(dummy_images.size(0), -1)
linear_img_model = nn.Linear(32*32, 10)

# Forward pass
out = linear_img_model(dummy_images_flat)
print("Output shape:", out.shape)

Output shape: torch.Size([2, 10])


: 

## Section 5 – Convolutional Neural Network (CNN)

- CNNs process images in their natural shape `[B, C, H, W]`.
- Convolution uses small **kernels (filters)** that slide across the image.
- Each kernel learns to detect a **specific visual pattern** (edges, textures, shapes).
- The same kernel is reused across the whole image (**weight sharing**), which **greatly reduces the number of parameters** compared to fully connected layers.
- Each convolution produces a **feature map** showing where a learned pattern appears in the image.
- Deeper layers combine simple patterns into **more complex visual features**.
- Designing CNNs from scratch requires tracking how both **the number of channels** and **spatial dimensions (H, W)** change after each layer.

---

### Computing the output size of a convolution

To compute the spatial size after a convolution layer, we use:

$$
H_{out} = \left\lfloor \frac{H_{in} + 2P - K}{S} \right\rfloor + 1
$$

Where:

- $H_{in}$ – input height (same formula for width)
- $K$ – kernel size
- $S$ – stride (determines the jumps of the kernel)
- $P$ – padding


```
out = floor((W + 2P - K) /S) + 1
```

The same formula applies to the width dimension.


In [ ]:
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # input:  [B, 1, 32, 32]
        # Conv2d(in_channels=1, out_channels=8, kernel=3, stride=1)
        # spatial: (32 - 3) / 1 + 1 = 30
        # output: [B, 8, 30, 30]
        self.conv1 = nn.Conv2d(1, 8, 3, 1)
        self.conv2 = nn.Conv2d(8, 16, 3, 1)

        self.fc1 = nn.Linear(16*28*28, 32)
        self.fc2 = nn.Linear(32, 10)

    def forward(self, x):

        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))

        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

cnn_model = SimpleCNN()
dummy_out = cnn_model(dummy_images)
print("CNN output shape:", dummy_out.shape)

CNN output shape: torch.Size([2, 10])


: 

In [ ]:
# from torchinfo import summary
# summary(cnn_model, input_size=(1,1,32,32))

: 

## Section 6 – Autoencoder Concept

- Autoencoders compress input into a **latent space** (encoder) and reconstruct (decoder).
- Useful to show CNNs for **reconstruction/generation**, not only classification.
- Transposed convolutional layers perform up-sampling from a low resolution feature space to a high resolution output.

In [ ]:
class SimpleAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder
        self.enc1 = nn.Conv2d(1, 8, 3, 2, 1)  
        # H_out = floor((H + 2p - k)/s) + 1 = floor((H + 2*1 - 3)/2) + 1 ≈ H/2

        self.enc2 = nn.Conv2d(8, 4, 3, 2, 1)

        # Decoder
        self.dec1 = nn.ConvTranspose2d(4, 8, 3, 2, 1, 1)  
        # H_out = (H-1)*s - 2p + k + op = (H-1)*2 - 2 + 3 + 1 = 2H

        self.dec2 = nn.ConvTranspose2d(8, 1, 3, 2, 1, 1)

    def forward(self, x):
        x = F.relu(self.enc1(x))
        x = F.relu(self.enc2(x))
        x = F.relu(self.dec1(x))
        x = torch.sigmoid(self.dec2(x))
        return x


: 

### Why Autoencoders?

Autoencoders are an example of **representation learning**.

Instead of predicting a label, the network learns a **compressed representation of the input data** (called a *latent space*).

This encoder–decoder pattern appears in many important machine learning systems.

Examples:

**Image compression**  
The encoder learns a compact representation of an image that can later be reconstructed.

**Denoising**  
The network learns to reconstruct a clean image from a noisy one.

**Super-resolution**  
A model reconstructs a higher-resolution image from a low-resolution input.

**Image generation**  
Many generative models use a latent space that is decoded into images (e.g., VAEs, diffusion models).

**Feature extraction**  
The latent representation can be used as input for other tasks such as classification or clustering.

In this exercise we will **not train the model**, but we will build the **architecture** and inspect how data flows through an encoder–decoder network.

# Final Exercise – Build & Inspect Your Own Model

## Basic Level – MNIST Digit Classification Model

Build a neural network for **image classification using MNIST**, that predicts the **digit class (0–9)**.
The goal of this exercise is **building and inspecting a neural network**, not training it. 

Input:
MNIST image  
shape: `[batch_size, 1, 28, 28]`

Output:
digit prediction (0–9)  
shape: `[batch_size, 10]`

Compare **2 architectures**:
- linear model
- CNN

Steps:

1. Define your model architecture.
2. Count number of parameters of your model.
3. Run a **forward pass** on:
    - dummy image (1, 28, 28)
    - example batch from MNIST (from Dataloader)
    - example element of Dataset MNIST
    - all MNIST data (iterate over the dataset using a `DataLoader`) and calculate accuracy

Example:

```python
correct = 0
total = 0

for images, labels in train_loader:

    outputs = model(images)
    preds = outputs.argmax(dim=1)

    correct += (preds == labels).sum().item()
    total += labels.size(0)

accuracy = correct / total
print("Accuracy:", accuracy)
```

4. Inspect the model, print:
    - input tensor shape
    - output tensor shape
    - number of parameters

Example:

```python
sum(p.numel() for p in model.parameters())
```
5. Change the goal of the Model and run a **forward pass** as in previous step.

Imagine that instead of predicting digits (0–9) we would like to directly predict whether the digit is **even or odd**.

New output:

```
[batch_size, 2]
```

You will need to convert MNIST labels:

```python
labels_even_odd = labels % 2
```
6. Send your model to device (gpu or cpu).


### 0. Data loading and preprocessing

In [2]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_subset = Subset(train_dataset, range(100))
test_subset = Subset(test_dataset, range(30))

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=64, shuffle=False)

test_iter = iter(test_loader)
images, labels = next(test_iter)
print(images.shape)

torch.Size([30, 1, 28, 28])


### 1. Define the model architecture

In [29]:
import torch.nn.functional as F
import torch.nn as nn

class MNIST_CNN(nn.Module):
    def __init__(self):
        super().__init__()

        # input:  [B, 1, 28, 28]
        # Conv2d(in_channels=1, out_channels=8, kernel=3, stride=1)
        # spatial: (28 - 3) / 1 + 1 = 26
        # output: [B, 8, 26, 26]
        self.conv1 = nn.Conv2d(1, 8, 3, 1)
        # spatial: (26 - 3) / 1 + 1 = 24
        self.conv2 = nn.Conv2d(8, 16, 3, 1)

        self.fc1 = nn.Linear(16*24*24, 32)
        self.fc2 = nn.Linear(32, 10)

    def forward(self, x):

        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))


        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x


cnn_model = MNIST_CNN()
linear_model = nn.Linear(28*28, 10)

### 2. Count number of parameters of your model

In [23]:
# linear
print("Linear parameters")
lin_dict = linear_model.state_dict()
for key in lin_dict:
    print(f"{key}:", lin_dict[key].shape)

# cnn
print("")
print("CNN parameters")
cnn_dict = cnn_model.state_dict()
for key in cnn_dict:
    print(f"{key} size:", cnn_dict[key].shape)

Linear parameters
weight: torch.Size([10, 784])
bias: torch.Size([10])

CNN parameters
conv1.weight size: torch.Size([8, 1, 3, 3])
conv1.bias size: torch.Size([8])
conv2.weight size: torch.Size([16, 8, 3, 3])
conv2.bias size: torch.Size([16])
fc1.weight size: torch.Size([32, 9216])
fc1.bias size: torch.Size([32])
fc2.weight size: torch.Size([10, 32])
fc2.bias size: torch.Size([10])


### 3. Run a forward pass

In [ ]:
# dummy image (1, 28, 28)

X_a = torch.randn(1, 1, 28, 28)
y_a_lin = linear_model(X_a.flatten())
X_a = torch.randn(1, 1, 28, 28)
y_a_cnn = cnn_model.forward(X_a)

print(f"LINEAR:\t\t\tCNN:")
print(f"{y_a_lin.shape}\t{y_a_cnn.shape}")
for i in range(10):
    print(f"{y_a_lin[i]}\t{y_a_cnn[0, i]}")

LINEAR:			CNN:
torch.Size([10])	torch.Size([1, 10])
-0.18367668986320496	0.03342478722333908
0.5856656432151794	-0.09020484238862991
-1.257783055305481	-0.048085302114486694
-0.3663689196109772	-0.10994905978441238
0.5607912540435791	0.04451528936624527
0.18341569602489471	0.08671057224273682
-1.6483030319213867	-0.10087893903255463
-0.2847505211830139	0.011982142925262451
0.0075547341257333755	-0.06874974817037582
0.25460880994796753	0.08816970139741898


In [31]:
# example batch from MNIST (from Dataloader)
images_flat = images.view(images.size(0), -1)
y_b_lin = linear_model(images_flat)
y_b_cnn = cnn_model.forward(images)

print(f"LINEAR:\t\t\tCNN:")
print(f"{y_b_lin.shape}\t{y_b_cnn.shape}")

LINEAR:			CNN:
torch.Size([30, 10])	torch.Size([30, 10])


In [35]:
# example element of Dataset MNIST

image_flat = images[15].flatten()
y_c_lin = linear_model(image_flat)
y_c_cnn = cnn_model.forward(images[15].unsqueeze(0))

print(f"LINEAR:\t\t\tCNN:")
print(f"{y_c_lin.shape}\t{y_c_cnn.shape}")
for i in range(10):
    print(f"{y_c_lin[i]}\t{y_c_cnn[0, i]}")

LINEAR:			CNN:
torch.Size([10])	torch.Size([1, 10])
-0.23386326432228088	-0.0027177687734365463
0.2748772203922272	-0.06187395751476288
-0.5424286723136902	-0.018447812646627426
0.6335919499397278	-0.18258272111415863
-0.6661429405212402	0.03871604800224304
1.141088843345642	0.07590039074420929
-0.4201897084712982	-0.09231637418270111
0.24181413650512695	-0.03713972494006157
1.2800244092941284	-0.02557593211531639
0.46247607469558716	0.13343630731105804


In [44]:
# all MNIST data (iterate over the dataset using a `DataLoader`) and calculate accuracy
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
test_subset = Subset(test_dataset, range(100))
test_loader = DataLoader(test_subset, batch_size=20, shuffle=False)
test_iter = iter(test_loader)

acc_lin = 0  # This will accumulate total correct predictions for linear model
acc_cnn = 0  # This will accumulate total correct predictions for CNN model
processed = 0  # This will accumulate total samples processed

for images, labels in test_iter:
    bs = images.shape[0]
    y_d_lin = linear_model(images.view(images.size(0), -1))
    y_d_cnn = cnn_model.forward(images)
    
    # Get predicted classes (argmax over the output dimension, assuming shape [batch_size, num_classes])
    pred_lin = torch.argmax(y_d_lin, dim=1)
    pred_cnn = torch.argmax(y_d_cnn, dim=1)
    
    # Count correct predictions
    correct_lin = (pred_lin == labels).sum().item()
    correct_cnn = (pred_cnn == labels).sum().item()
    
    # Accumulate correct counts and total samples
    acc_lin += correct_lin
    acc_cnn += correct_cnn
    processed += bs

print("Accuracies:")
print(f"Linear: {acc_lin/processed}\t\tCNN: {acc_cnn/processed}")

Accuracies:
Linear: 0.04		CNN: 0.08


### 4. Inspect the model

In [ ]:
print("\t\t\tLinear:\t\tCNN:")
# input shape
cnn_model.
# output shape

# nr of parameters
print(f"{sum(p.numel() for p in linear_model.parameters())}\t\t{sum(p.numel() for p in cnn_model.parameters())}")

Linear:		CNN:
7850		296522


### 5. Change a goal of the model and tun a forward pass again

: 

### 6. Send your model to device (cpu or gpu)

: 

---

## Deep Level – Beyond Classification

Neural networks are **not limited** to classification. 

Instead of predicting a label, build a network that **reconstructs the input image**.

Architecture idea: Autoencoder

```
image → encoder → latent space → decoder → reconstructed image
```

Input:

```
[batch_size, 1, 28, 28]
```

Output:

```
[batch_size, 1, 28, 28]
```

Steps:

1. Build an **autoencoder**-like architecture that compresses the image and then reconstruct it.
2. Run a forward pass.
3. Inspect:
   - latent representation shape
   - reconstructed image shape
   - reconstructed image itself (`plt.imshow(output[0, 0].detach().cpu())`)
   - parameter count
